# Phase 1 — Data Ingestion

This notebook exercises `src/ingest.py` to pull every data source we'll need for the FanTeasy Stats pipeline. Each fetch is cached to `data/raw/` so re-running is fast.

**What we pull:**

| Source | What it gives us | Used in phase |
|---|---|---|
| nflverse play-by-play | Every play with air yards, pass location, etc. | 2 (features) + 5 (heatmaps) |
| nflverse weekly stats | Per-week fantasy totals per player | 2 + 6 |
| nflverse snap counts | Offensive/ST snap participation | 2 + 3 (roles) |
| nflverse NGS | aDOT, separation, time-to-throw | 2 + 4 (radar) |
| nflverse schedule | Weather, roof, matchup context | 6 (projections) |
| nflverse rosters | Player-team-season assignments | 2 |
| nflverse ID crosswalk | gsis_id ↔ sleeper_id joiner | 7 (export) |
| Sleeper players | Injury status, depth chart, sleeper_id | 7 |
| Sleeper league | Scoring settings, roster positions | 2 (custom scoring) |
| Sleeper projections | Baseline to beat with our model | 6 |


## Setup

In [ ]:
# Add project root to path so we can import from src/
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Verbose logging so we can see cache hits/misses
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [ ]:
from src.ingest import (
    get_pbp, get_weekly_stats, get_snap_counts,
    get_ngs_data, get_schedule, get_seasonal_rosters,
    get_id_crosswalk,
    get_sleeper_league, get_sleeper_players, get_sleeper_projections,
    DEFAULT_LEAGUE_ID,
)

## Choose which season(s) to pull

For a single-season model, 1 year is enough. For anything predictive it's worth pulling at least 2 so the model sees week-to-week variance across years.

> **Note**: `nflreadpy` season data becomes available a few days after each week's games. If the current season is very early or hasn't started yet, only pull past seasons.

In [ ]:
# Adjust as the season progresses. 2024 = full historical, 2025 = current.
SEASONS = [2024, 2025]

## 1. Sleeper league config

Small fetch, but everything else depends on knowing the league's scoring rules and roster structure.

In [ ]:
league = get_sleeper_league(DEFAULT_LEAGUE_ID)
print(f"League: {league.get('name')}  ({league.get('season')} season)")
print(f"Teams: {league.get('total_rosters')}")
print(f"Scoring format: {league.get('scoring_settings', {}).get('rec', 0)}pt PPR")
print(f"Roster slots: {league.get('roster_positions')}")

# Sleeper mints a new league_id per season for dynasty leagues. If the season
# above isn't the one you're modeling, this is last year's league — grab the
# current id from Sleeper and update DEFAULT_LEAGUE_ID in src/ingest.py.
print(f"\nprevious_league_id: {league.get('previous_league_id')}")

In [ ]:
# Inspect the full scoring settings — this drives custom fantasy point
# calculations in Phase 2. Any field here has a stat with the same name
# in the weekly/pbp data.
scoring = league.get('scoring_settings', {})
scoring_df = pd.DataFrame([
    {'setting': k, 'value': v}
    for k, v in scoring.items() if isinstance(v, (int, float)) and v != 0
]).sort_values('setting')
print(f"{len(scoring_df)} non-zero scoring settings:")
scoring_df

## 2. Sleeper player DB

The full NFL player registry from Sleeper's side. Provides the `sleeper_id` we'll key everything by in the final JSON export.

In [ ]:
sleeper_players = get_sleeper_players()
print(f"Total Sleeper player entries: {len(sleeper_players):,}")
sleeper_players.head()

In [ ]:
# Quick sanity checks — how many active fantasy-relevant players?
active_fantasy = sleeper_players[
    (sleeper_players['position'].isin(['QB', 'RB', 'WR', 'TE', 'K', 'DEF']))
    & (sleeper_players['status'] == 'Active')
]
print(f"Active fantasy-relevant players: {len(active_fantasy):,}")
print(active_fantasy['position'].value_counts().to_string())

## 3. Player ID crosswalk

The **most important** table for later phases. nflverse's `gsis_id` and Sleeper's `sleeper_id` don't match — this table bridges them plus a handful of other systems (ESPN, Yahoo, PFR, PFF).

In [ ]:
crosswalk = get_id_crosswalk()
print(f"Total crosswalk rows: {len(crosswalk):,}")
print(f"Columns available: {list(crosswalk.columns)}")
crosswalk.head()

In [ ]:
# How complete is the sleeper_id column? Some historical players
# without Sleeper accounts will have NaN — that's fine, they can't be
# in our export anyway.
with_sleeper = crosswalk['sleeper_id'].notna().sum()
print(f"Rows with sleeper_id: {with_sleeper:,} / {len(crosswalk):,}")
print(f"Rows with gsis_id:    {crosswalk['gsis_id'].notna().sum():,}")

# Save a slim version we'll use for join lookups later
from src.ingest import DATA_PROCESSED
slim = crosswalk[['gsis_id', 'sleeper_id', 'name', 'position', 'team']].dropna(subset=['sleeper_id'])
out_path = DATA_PROCESSED / 'id_crosswalk_slim.csv'
slim.to_csv(out_path, index=False)
print(f"\nSlim crosswalk saved: {out_path}  ({len(slim):,} rows)")

## 4. Weekly stats (aggregated by player-week)

Easier to work with than raw pbp for most feature computations. Use this for anything that doesn't need play-level detail.

In [ ]:
weekly = get_weekly_stats(SEASONS)
print(f"Weekly stat rows: {len(weekly):,}")
print(f"Columns: {list(weekly.columns)}")
weekly.head()

In [ ]:
# Spot-check: highest fantasy scorers in the most recent complete week
latest_season = max(SEASONS)
latest_wk = weekly[weekly['season'] == latest_season]['week'].max()
print(f"Latest week in data: {latest_season} Wk {latest_wk}")

# nflverse renamed this column between stat releases ('recent_team' -> 'team').
# Resolve whichever exists so this cell survives the next schema bump.
TEAM_COL = next(c for c in ['team', 'recent_team'] if c in weekly.columns)
print(f"Using team column: {TEAM_COL}")

top10 = (weekly[(weekly['season'] == latest_season) & (weekly['week'] == latest_wk)]
         .sort_values('fantasy_points_ppr', ascending=False)
         .head(10)[['player_display_name', 'position', TEAM_COL,
                    'fantasy_points_ppr', 'fantasy_points']])
top10

## 5. Play-by-play

The big one — ~50k rows per season. Only fetch this if you need play-level fields (air_yards, pass_location, run_gap, etc.) which you'll need for radar metrics and heatmaps.

In [ ]:
pbp = get_pbp(SEASONS)
print(f"Play-by-play rows: {len(pbp):,}")
# Skip printing all columns — there are 300+ — just show the ones
# we'll actually use in Phase 2
phase2_cols = [c for c in pbp.columns if c in [
    'passer_player_id', 'receiver_player_id', 'rusher_player_id',
    'passing_yards', 'receiving_yards', 'rushing_yards',
    'pass_touchdown', 'rush_touchdown',
    'air_yards', 'yards_after_catch',
    'pass_location', 'run_location', 'run_gap',
    'yardline_100', 'complete_pass', 'interception', 'sack',
    'week', 'season', 'posteam', 'defteam',
]]
pbp[phase2_cols].head()

## 6. Snap counts

Critical for role classification: is this RB a 3-down back (65%+ snap share) or a committee back?

In [ ]:
snaps = get_snap_counts(SEASONS)
print(f"Snap-count rows: {len(snaps):,}")
snaps.head()

## 7. Next Gen Stats

aDOT, separation, time-to-throw. Only available for players with tracking-data participation, so smaller universe than pbp.

In [ ]:
ngs_receiving = get_ngs_data('receiving', SEASONS)
ngs_passing = get_ngs_data('passing', SEASONS)
ngs_rushing = get_ngs_data('rushing', SEASONS)

print(f"NGS receiving rows: {len(ngs_receiving):,}")
print(f"NGS passing rows:   {len(ngs_passing):,}")
print(f"NGS rushing rows:   {len(ngs_rushing):,}")

## 8. Schedule (weather + matchup context)

Small table but useful for the projection model — dome vs outdoor, temperature, wind.

In [ ]:
schedule = get_schedule(SEASONS)
print(f"Schedule rows: {len(schedule):,}")
schedule[['game_id', 'season', 'week', 'gameday', 'home_team', 'away_team',
          'roof', 'surface', 'temp', 'wind']].head(10)

## 9. Sleeper projections

The baseline we're trying to beat. Fetches per-week — pull the current week for reference and any past week you want to backtest against.

In [ ]:
# Pick a week — adjust to whatever's current for your project
PROJ_SEASON = 2025
PROJ_WEEK = 1

proj = get_sleeper_projections(PROJ_SEASON, PROJ_WEEK)
print(f"Projection rows for {PROJ_SEASON} Wk {PROJ_WEEK}: {len(proj):,}")
print(f"Columns available: {list(proj.columns)[:20]}...")
proj.head()

## Sanity: end-to-end join test

Pick one player from Sleeper's DB and confirm we can join them to their nflverse stats via the crosswalk. If this works for one player, it works for all of them.

In [ ]:
# Look up a well-known player (e.g. Josh Allen) by Sleeper ID
sample_sleeper_id = sleeper_players[
    sleeper_players['full_name'].str.contains('Josh Allen', na=False)
    & (sleeper_players['position'] == 'QB')
]['sleeper_id'].iloc[0]
print(f"Josh Allen's sleeper_id: {sample_sleeper_id!r}")

# Both sides are strings because get_id_crosswalk() normalizes ID columns.
# If this assert trips, that normalization is the first thing to check —
# a float 4984.0 will never match the string '4984'.
match = crosswalk[crosswalk['sleeper_id'] == sample_sleeper_id]
assert not match.empty, "Crosswalk miss — check sleeper_id dtype on both sides"
gsis = match['gsis_id'].iloc[0]
print(f"Josh Allen's gsis_id:    {gsis!r}")

# Pull all their weekly rows from nflverse
wanted = ['season', 'week', 'player_display_name', TEAM_COL,
          'completions', 'attempts', 'passing_yards', 'passing_tds',
          'interceptions', 'carries', 'rushing_yards', 'rushing_tds',
          'fantasy_points_ppr']
player_weekly = (weekly[weekly['player_id'] == gsis]
                 [[c for c in wanted if c in weekly.columns]]
                 .sort_values(['season', 'week']))
print(f"\nRows found in weekly stats: {len(player_weekly)}")
player_weekly.tail(10)

## What's next

Everything above should now be sitting in `data/raw/` as parquet/json files. Re-runs of any of these functions will hit the cache and return instantly.

**Next step — Phase 2 (Feature Engineering):**
- Aggregate pbp into per-player-per-week features (aDOT, YAC, snap share, target share, etc.)
- Apply the league's custom scoring rules to compute true fantasy points
- Handle position-specific feature sets

See `notebooks/02_feature_engineering.ipynb` (coming next).